# Brian2 Simulation from SONATA

End-to-end example that runs the *Drosophila* (FlyWire) connectome through obi-one's
Brian2 SONATA pipeline and checks the spiking activity against the original
reference model:

1. **Convert** the `philshiu/Drosophila_brain_model` connectome to a SONATA
   point-neuron circuit with
   [`examples/J_drosophila/drosophila_to_brian2_sonata.py`](../J_drosophila/drosophila_to_brian2_sonata.py).
   `convert()` writes `nodes.h5`, `edges.h5`, the Brian2 model templates
   (`models/drosophila.json`, `models/synapse.json`), `node_sets.json`
   (including the `sugar` neuron set) and `circuit_config.json`.
2. **Generate** the SONATA simulation campaign with obi-one's
   `Brian2CircuitSimulationScanConfig`
   (`obi_one/scientific/tasks/generate_simulations/config/brian2/brian2_circuit.py`).
   `CoupledScanGenerationTask` → `GenerateSimulationTask` sweeps over the random
   seeds and writes one `simulation_config.json` per coordinate with
   `target_simulator: "Brian2"`; the sugar drive is declared as a
   `Brian2DirectPoissonStimulus` block so the generated config already carries
   the `poisson` input.
3. **Run** each generated `simulation_config.json` with the SONATA → Brian2
   runner `obi_one/scientific/library/simulation/brian2/simulate-brian2.py`
   (`run_sonata_brian2_trial`), which loads the SONATA circuit through
   bluepysnap/libsonata, builds the Brian2 `NeuronGroup`/`Synapses`, expands the
   `poisson` input into one `PoissonInput` per `sugar` neuron and writes a
   SONATA `spikes.h5`.
4. **Validate** the result against the unmodified `model.run_trial` from the
   reference repository — the two flows are configured identically so the spike
   output is reproduced *bitwise*.

Each trial (both the SONATA runner and the reference) is executed in a fresh
subprocess so Brian2's global state (magic network, compiled modules, RNG
bookkeeping) cannot drift between runs.

In [1]:
import subprocess  # noqa: F401  (used by the run/reference cells below)
import sys

# brian2 ships with the obi-one environment, so the auto-install below is disabled.
# import importlib.util
# if importlib.util.find_spec('brian2') is None:
#     subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'brian2'])

In [2]:
import json
import logging
import pickle
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HERE = Path.cwd()
REPO_ROOT = HERE.parents[1]
J_DROSOPHILA = REPO_ROOT / 'examples' / 'J_drosophila'
SIMULATE_BRIAN2 = (
    REPO_ROOT / 'obi_one' / 'scientific' / 'library' / 'simulation' / 'brian2'
    / 'simulate_brian2.py'
)
REFERENCE_HELPER = (HERE / '_reference_trial.py').resolve()

CIRCUIT_DIR = HERE / 'sonata_circuit'
CAMPAIGN_DIR = HERE / 'obi_simulation_output' / 'campaign'

assert SIMULATE_BRIAN2.exists(), f'{SIMULATE_BRIAN2} not found'
assert REFERENCE_HELPER.exists(), f'{REFERENCE_HELPER} not found'
assert (J_DROSOPHILA / 'drosophila_to_brian2_sonata.py').exists()

print(f'Converter:     {J_DROSOPHILA / "drosophila_to_brian2_sonata.py"}')
print(f'SONATA runner: {SIMULATE_BRIAN2}')
print(f'Reference:     {REFERENCE_HELPER}')
print(f'Circuit dir:   {CIRCUIT_DIR}')

Converter:     /Users/james/Documents/obi/code/obi-main/obi-one/examples/J_drosophila/drosophila_to_brian2_sonata.py
SONATA runner: /Users/james/Documents/obi/code/obi-main/obi-one/obi_one/scientific/library/simulation/brian2/simulate_brian2.py
Reference:     /Users/james/Documents/obi/code/obi-main/obi-one/examples/J_drosophila_brian2_sonata/_reference_trial.py
Circuit dir:   /Users/james/Documents/obi/code/obi-main/obi-one/examples/J_drosophila_brian2_sonata/sonata_circuit


## 2. Convert the connectome to SONATA

Import the converter module and the reference `default_params` (network constants
and Brian2 equations), then point at the already-converted **630** SONATA circuit
in `examples/J_drosophila/output-630` — built by running
`drosophila_to_brian2_sonata.py`, whose `convert()` writes `nodes.h5`, `edges.h5`,
the Brian2 model templates (`models/drosophila.json`, `models/synapse.json`),
`node_sets.json` and `circuit_config.json`. The `sugar` neuron set — the 21
FlyWire neurons that receive the Poisson drive — is written into `node_sets.json`
by the converter.

In [3]:
# Local clone of philshiu/Drosophila_brain_model: provides model.py (run_trial,
# default_params) and the 630-dataset completeness/connectivity files.
DROSOPHILA_REPO = J_DROSOPHILA / 'Drosophila_brain_model'

if str(J_DROSOPHILA) not in sys.path:
    sys.path.insert(0, str(J_DROSOPHILA))
if str(DROSOPHILA_REPO) not in sys.path:
    sys.path.insert(0, str(DROSOPHILA_REPO))

import drosophila_to_brian2_sonata as dros_sonata  # noqa: F401  (converter; see markdown)
from brian2 import Hz, mV
from model import default_params

SIM_LENGTH_MS = 200.0
SIM_DT_MS = 0.1  # Brian2's default integration step; the reference uses the same
SEEDS = [42, 43]
POISSON_RATE_HZ = float(default_params['r_poi'] / Hz)  # 150 Hz
# Poisson synapse weight from the reference model: w_syn * f_poi = 0.275 mV * 250.
# simulate_brian2.py applies it as `weight * mV`, so pass the millivolt value.
POISSON_WEIGHT = float(default_params['w_syn'] * default_params['f_poi'] / mV)  # 68.75
V_INIT_MV = -52.0  # reference resting/reset potential (v_0 = v_rst)

# The 21 sugar-sensing FlyWire neurons (same list as the converter's __main__).
SUGAR_NODES = [
    720575940611875570, 720575940612670570, 720575940616885538,
    720575940617000768, 720575940617937543, 720575940620900446,
    720575940621502051, 720575940621754367, 720575940624963786,
    720575940628853239, 720575940629176663, 720575940630233916,
    720575940630797113, 720575940632425919, 720575940632889389,
    720575940633143833, 720575940637568838, 720575940638202345,
    720575940639198653, 720575940639332736, 720575940640649691,
]

# 630 dataset — the converter's __main__ default and the dataset _reference_trial.py
# is configured for. The completeness CSV defines the neuron ordering, so the SONATA
# node indices (and the `sugar` node set) line up with the reference's brian indices.
CIRCUIT_DIR = J_DROSOPHILA / 'output-630'
PATH_COMP = DROSOPHILA_REPO / '2023_03_23_completeness_630_final.csv'
circuit_config_path = CIRCUIT_DIR / 'circuit_config.json'
assert circuit_config_path.exists(), (
    f'{circuit_config_path} not found — build it by running '
    f'`python drosophila_to_brian2_sonata.py` in {J_DROSOPHILA}'
)

sugar_indices = json.loads(
    (CIRCUIT_DIR / 'node_sets.json').read_text()
)['sugar']['node_id']

N_NEURONS = len(pd.read_csv(PATH_COMP, index_col=0))
print(f'SONATA circuit:  {circuit_config_path}')
print(f'neurons:         {N_NEURONS}')
print(f'sugar neurons:   {len(sugar_indices)}')
print(f'Poisson drive:   {POISSON_RATE_HZ:g} Hz, weight {POISSON_WEIGHT:g} mV')

SONATA circuit:  /Users/james/Documents/obi/code/obi-main/obi-one/examples/J_drosophila/output-630/circuit_config.json
neurons:         127400
sugar neurons:   21
Poisson drive:   150 Hz, weight 68.75 mV


## 3. Generate the SONATA simulation campaign with obi-one

Build a `Brian2CircuitSimulationScanConfig` pointed at the converted circuit,
sweep over the random seeds, and run `GenerateSimulationTask` through
`CoupledScanGenerationTask` to produce one `simulation_config.json` per
coordinate. The sugar drive is declared as a `Brian2DirectPoissonStimulus`
block, so the generated SONATA config already contains the `poisson` input — no
hand-written JSON.

`initialize._timestep` is pinned to 0.1 ms (Brian2's default, which the runner
uses) and re-asserted on each generated single config, because `_timestep` is a
`PrivateAttr` that is dropped when the scan machinery re-creates the config via
`model_dump`/`model_validate`.

In [4]:
from obi_one.core.info import Info
from obi_one.core.run_tasks import run_task_for_single_configs
from obi_one.core.scan_generation import CoupledScanGenerationTask
from obi_one.core.tuple import NamedTuple
from obi_one.scientific.blocks.neuron_sets.id import IDNeuronSet
from obi_one.scientific.blocks.neuron_sets.specific import AllNeurons
from obi_one.scientific.blocks.stimuli.brian2_poisson import Brian2DirectPoissonStimulus
from obi_one.scientific.library.circuit import Circuit
from obi_one.scientific.tasks.generate_simulations.config.brian2.brian2_circuit import (
    Brian2CircuitSimulationScanConfig,
)

In [ ]:
circuit = Circuit(name='drosophila', path=str(circuit_config_path.absolute()))
print(f'circuit loaded: {circuit.sonata_circuit.nodes["drosophila"].size} neurons')

sim_conf = Brian2CircuitSimulationScanConfig.empty_config()
sim_conf.set(
    Info(
        campaign_name='Brian2 fly demo',
        campaign_description=f'{SIM_LENGTH_MS:.0f} ms Brian2 simulation of the fly brain',
    ),
    name='info',
)
all_neurons = AllNeurons()
sim_conf.add(all_neurons, name='AllNeurons')

sugar_drive_set = IDNeuronSet(
    neuron_ids=NamedTuple(name='sugar', elements=tuple(int(i) for i in sugar_indices)),
)
sim_conf.add(sugar_drive_set, name='SugarDrive')

sim_conf.add(
    Brian2DirectPoissonStimulus(
        neuron_set=sugar_drive_set.ref,
        frequency=POISSON_RATE_HZ,
        weight=POISSON_WEIGHT,
        duration=SIM_LENGTH_MS,
    ),
    name='poisson_sugar',
)

initialize = Brian2CircuitSimulationScanConfig.Initialize(
    circuit=circuit,
    simulation_length=SIM_LENGTH_MS,
    random_seed=SEEDS,
    v_init=V_INIT_MV,
)
initialize._timestep = SIM_DT_MS
sim_conf.set(initialize, name='initialize')

validated = sim_conf.validated_config()
print(validated)

circuit loaded: 127400 neurons
Brian2CircuitSimulationScanConfig(type='Brian2CircuitSimulationScanConfig', info=Info(type='Info', campaign_name='Brian2 fly demo', campaign_description='200 ms Brian2 simulation of the fly brain'), initialize=Initialize(type='Brian2CircuitSimulationScanConfig.Initialize', simulation_length=200.0, v_init=-52.0, random_seed=[42, 43], circuit=Circuit(name='drosophila', path='/Users/james/Documents/obi/code/obi-main/obi-one/examples/J_drosophila/output-630/circuit_config.json', matrix_path=None, type='Circuit'), node_set=None), stimuli={'poisson_sugar': Brian2DirectPoissonStimulus(type='Brian2DirectPoissonStimulus', neuron_set=NeuronSetReference(block_dict_name='neuron_sets', block_name='AllNeurons', type='NeuronSetReference'), frequency=150.0, weight=68.75, duration=200.0)}, neuron_sets={'AllNeurons': AllNeurons(type='AllNeurons', sample_percentage=100.0, sample_seed=1), 'SugarDrive': IDNeuronSet(type='IDNeuronSet', sample_percentage=100.0, sample_seed=1, n

In [6]:
coupled_scan = CoupledScanGenerationTask(form=validated, output_root=str(CAMPAIGN_DIR))
coupled_scan.multiple_value_parameters(display=True)
coupled_scan.execute()

# _timestep is a PrivateAttr, dropped when the scan machinery re-creates the
# config; re-assert it before GenerateSimulationTask reads it into run.dt.
for sc in coupled_scan.single_configs:
    sc.initialize._timestep = SIM_DT_MS

run_task_for_single_configs(single_configs=coupled_scan.single_configs)

single_configs = coupled_scan.single_configs
coord_roots = [Path(sc.coordinate_output_root) for sc in single_configs]
print(f'\nGenerated {len(coord_roots)} simulation coordinate(s):')
for cr in coord_roots:
    print(f'  {cr}')

[2026-06-23 11:59:26,259] INFO: 
MULTIPLE VALUE PARAMETERS
[2026-06-23 11:59:26,259] INFO: initialize.random_seed: [42, 43]
[2026-06-23 11:59:26,263] INFO: initialize.circuit is a Circuit instance.
[2026-06-23 11:59:26,263] INFO: initialize.node_set is None — setting default node set.


ValueError: Number of neurons used with the Direct Poisson Input exceeds the maximum allowed: 100.

In [ ]:
with (coord_roots[0] / 'simulation_config.json').open() as f:
    generated = json.load(f)

print(f'target_simulator: {generated["target_simulator"]}')
print(f'run:              {generated["run"]}')
print(f'network:          {generated["network"]}')
print(f'inputs:           {list(generated.get("inputs", {}).keys())}')
print(f'poisson input:    {generated["inputs"]["poisson_sugar"]}')

## 4. Run each generated config with the SONATA → Brian2 runner

`simulate-brian2.py sonata-simulation` calls `run_sonata_brian2_trial`, which
seeds Brian2 from `run.random_seed`, builds the network from the SONATA circuit
and writes `spikes.h5` under the config's `output` directory. Running it as a
subprocess gives each seed a pristine Brian2 interpreter.

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

spike_files = {}
for sc, coord_root in zip(single_configs, coord_roots):
    seed = sc.initialize.random_seed
    cfg = coord_root / 'simulation_config.json'
    print(f'\n--- seed {seed}: {coord_root.name} ---')
    subprocess.check_call([
        sys.executable, str(SIMULATE_BRIAN2), 'sonata-simulation',
        '--simulation-path', str(cfg), '-vv',
    ])
    spikes_h5 = coord_root / 'output' / 'spikes.h5'
    assert spikes_h5.exists(), spikes_h5
    spike_files[seed] = spikes_h5
    print(f'  -> {spikes_h5}')

## 5. Analyze the SONATA → Brian2 output

`run_sonata_brian2_trial` writes spike times in seconds; scale to milliseconds
for the rasters.

In [ ]:
def load_spikes(spikes_h5):
    with h5py.File(spikes_h5, 'r') as f:
        pop = next(iter(f['spikes']))
        node_ids = f[f'spikes/{pop}/node_ids'][:]
        timestamps_ms = f[f'spikes/{pop}/timestamps'][:] * 1000.0  # s -> ms
    return pop, node_ids, timestamps_ms


sonata_spikes = {seed: load_spikes(spike_files[seed]) for seed in SEEDS}

for seed in SEEDS:
    pop, ids, ts = sonata_spikes[seed]
    uniq, counts = np.unique(ids, return_counts=True)
    print(f'=== seed {seed} / {pop} ===')
    print(f'  active neurons: {len(uniq):,} / {N_NEURONS:,}')
    print(f'  total spikes:   {len(ids):,}')
    print(f'  mean rate over active neurons: '
          f'{counts.mean() / (SIM_LENGTH_MS / 1000.0):.2f} Hz')

In [ ]:
fig, axs = plt.subplots(len(SEEDS), 1, figsize=(12, 3.0 * len(SEEDS)),
                        sharex=True, squeeze=False)
for ax, seed in zip(axs[:, 0], SEEDS):
    _, ids, ts = sonata_spikes[seed]
    ax.scatter(ts, ids, s=0.3, alpha=0.5, c='black')
    ax.set_title(f'SONATA -> Brian2 spike raster (seed {seed})')
    ax.set_ylabel('Neuron ID')
axs[-1, 0].set_xlabel('Time (ms)')
axs[-1, 0].set_xlim(0, SIM_LENGTH_MS)
plt.tight_layout()
plt.show()

## 6. Validation against the reference `model.run_trial`

Re-run each seed with the unmodified `run_trial` from
`philshiu/Drosophila_brain_model/model.py` and confirm the obi-one pipeline
reproduces its spike output.

For a fair comparison the reference is configured exactly as the generated
campaign:

- same seed, simulation length, Poisson rate/weight
- `brian2.defaultclock.dt = 0.1 ms` — Brian2's default, which the SONATA runner
  also uses (the scan config pins `initialize._timestep = 0.1 ms`)
- sugar indices passed in sorted order, the same order in which the SONATA
  runner materializes the `sugar` node set, so the `PoissonInput`s consume the
  RNG stream identically
- `run_sonata_brian2_trial` creates the `SpikeMonitor` before the
  `PoissonInput`s, mirroring the reference `create_model -> poi` order
- each run is isolated in a fresh subprocess

In [ ]:
SUGAR_REFERENCE_INDICES = sorted(int(i) for i in sugar_indices)

reference_spike_trains = {}
for seed in SEEDS:
    print(f'--- reference run_trial seed={seed} (subprocess) ---')
    out_path = HERE / f'_reference_trial_seed_{seed}.pkl'
    subprocess.check_call([
        sys.executable, str(REFERENCE_HELPER), str(DROSOPHILA_REPO),
        str(SIM_LENGTH_MS), str(SIM_DT_MS), str(POISSON_RATE_HZ),
        str(seed), json.dumps(SUGAR_REFERENCE_INDICES), str(out_path),
    ])
    result = pickle.loads(out_path.read_bytes())
    out_path.unlink()

    trains = {}
    for nid, t in zip(result['node_ids'], result['times_s'], strict=False):
        trains.setdefault(int(nid), []).append(float(t))
    reference_spike_trains[seed] = {k: np.asarray(v) for k, v in trains.items()}

    n_active = len(reference_spike_trains[seed])
    n_spikes = sum(len(v) for v in reference_spike_trains[seed].values())
    print(f'  reference: {n_active} active, {n_spikes} spikes')

In [ ]:
def to_count_vector(node_ids, n_nodes):
    v = np.zeros(n_nodes, dtype=int)
    uniq, counts = np.unique(np.asarray(node_ids, dtype=np.int64),
                             return_counts=True)
    v[uniq] = counts
    return v


def ref_count_vector(trains, n_nodes):
    v = np.zeros(n_nodes, dtype=int)
    for idx, times in trains.items():
        v[int(idx)] = len(times)
    return v


rows = []
for seed in SEEDS:
    _, sonata_ids, _ = sonata_spikes[seed]
    obi_counts = to_count_vector(sonata_ids, N_NEURONS)
    ref_counts = ref_count_vector(reference_spike_trains[seed], N_NEURONS)

    active = (obi_counts > 0) | (ref_counts > 0)
    n_active = int(active.sum())
    identical = bool(np.array_equal(obi_counts, ref_counts))
    if n_active:
        r = float(np.corrcoef(obi_counts[active], ref_counts[active])[0, 1])
        delta = float(np.mean(np.abs(obi_counts - ref_counts)))
    else:
        r, delta = float('nan'), 0.0

    print(f'seed={seed}: sonata_total={obi_counts.sum()} '
          f'ref_total={ref_counts.sum()} active={n_active} '
          f'pearson_r={r:.4f} mean|delta|={delta:.3f} identical={identical}')
    rows.append({
        'seed': seed,
        'sonata_total': int(obi_counts.sum()),
        'ref_total': int(ref_counts.sum()),
        'active': n_active,
        'pearson_r': r,
        'mean_abs_delta': delta,
        'identical': identical,
    })

pd.DataFrame(rows)

In [ ]:
def raster_from_trains(trains, time_scale=1.0):
    ts, ids = [], []
    for idx, times in trains.items():
        arr = np.asarray(times, dtype=float) * time_scale
        ts.extend(arr.tolist())
        ids.extend([int(idx)] * len(arr))
    return np.asarray(ts), np.asarray(ids)


fig, axs = plt.subplots(len(SEEDS), 2, figsize=(13, 3.0 * len(SEEDS)),
                        sharex=True, sharey=True, squeeze=False)
for row, seed in enumerate(SEEDS):
    _, s_ids, s_ts = sonata_spikes[seed]
    axs[row, 0].scatter(s_ts, s_ids, s=0.5, alpha=0.6, c='black')
    axs[row, 0].set_title(f'obi-one SONATA -> Brian2 — seed {seed}')
    axs[row, 0].set_ylabel('neuron idx')

    r_ts, r_ids = raster_from_trains(reference_spike_trains[seed],
                                     time_scale=1000.0)  # s -> ms
    axs[row, 1].scatter(r_ts, r_ids, s=0.5, alpha=0.6, c='tab:red')
    axs[row, 1].set_title(f'reference run_trial — seed {seed}')

for ax in axs[-1, :]:
    ax.set_xlabel('time (ms)')
axs[-1, 0].set_xlim(0, SIM_LENGTH_MS)
plt.tight_layout()
plt.show()

The obi-one Brian2 pipeline — `drosophila_to_brian2_sonata.convert()` to build
the circuit, `Brian2CircuitSimulationScanConfig` /
`CoupledScanGenerationTask` to generate the `simulation_config.json`, and
`run_sonata_brian2_trial` to execute it — reproduces the reference
`model.run_trial` spike output **bitwise**: the table above shows
`identical = True` and `pearson_r = 1.0` for every seed.